In [ ]:
!pip install pandas==2.2.3 pinecone==6.0.2 -qU

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
from pinecone import Pinecone

In [ ]:
#Instance of pinecone client
pc = Pinecone(api_key = '')

In [ ]:
pc

With pinecone you can create a vector index/db where you can easily store and search through your vector embeddings

In [ ]:
# Giving our index a name
index_name = 'test'

In [ ]:
# Delete the index if an index of the same name already exists
if pc.has_index(name = index_name):
  pc.delete_index(name = index_name)

# Creating a pinecone index
When creating the index we need to define several configuration properties

- ```name``` can be any anything we like, the name is used as an identifier for the index when performating other operations such describe_index, delete_index and soon

- ```Metric``` specifies the similarity metric that will be used later when you make queries to the index

- ```dimension``` should correspond to the dimenstion of the dense vectors produced by your embedding model. In this quick start we are using made up data so small value is simplest

- ```spec``` holds a specification which tells Pinecone how you would like to deploy our index

In [ ]:
from pinecone import ServerlessSpec, CloudProvider, AwsRegion, Metric


pc.create_index(
    name = index_name,
    metric = Metric.COSINE, #cosine similarity search
    dimension = 3,
    spec = ServerlessSpec(cloud=CloudProvider.AWS, region = AwsRegion.US_EAST_1)
)



{
    "name": "test",
    "metric": "cosine",
    "host": "test-7gnb41n.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null
}

We can look up the configuration for the index anytime we like using ``` describe_index``` method

In [ ]:
description = pc.describe_index(name = index_name)
description

{
    "name": "test",
    "metric": "cosine",
    "host": "test-7gnb41n.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null
}

# Upserting/Inserting data into the index

We can see the index ready in the pinecone dashboar, now we will create some simple vectors that will serve as example data for inserting into pinecone db

In [ ]:
# Create instance of Index client
index = pc.Index(host = description.host)

In [ ]:
# Create dummy vector data

import random
import pandas as pd

def create_simulated_data_in_df(num_vectors):
  df = pd.DataFrame(
      data = {
          'id' : [f'id-{i}' for i in range(num_vectors)],
          'vector': [
              [random.random() for i in range(description.dimension)]
              for _ in range(num_vectors)
          ]
      }
  )
  return df

df = create_simulated_data_in_df(10)
df

,id,vector
0,id-0,"[0.19144389656116134, 0.7937812360800525, 0.45..."
1,id-1,"[0.6867595028860487, 0.949542163819325, 0.5555..."
2,id-2,"[0.6379964663477001, 0.14963514635676034, 0.16..."
3,id-3,"[0.6373580333294926, 0.046523173037209875, 0.6..."
4,id-4,"[0.7268641047916224, 0.058043738577178594, 0.5..."
5,id-5,"[0.7746189848758195, 0.7583960259612521, 0.051..."
6,id-6,"[0.1783787355170372, 0.9173900904089115, 0.591..."
7,id-7,"[0.31231160207149067, 0.47317447653817835, 0.6..."
8,id-8,"[0.05894786818101483, 0.6296337983665093, 0.65..."
9,id-9,"[0.8725454301603238, 0.6194616490470858, 0.992..."


We will now perform ```upsert``` operation in our index. This call will insert a new vector in the index or update the vector if the id was already present.

In [ ]:
index.upsert(vectors = zip(df.id, df.vector)) # Insert vectors

{'upserted_count': 10}

In [ ]:
# View some index stats

index.describe_index_stats()

{'dimension': 3,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 10}},
 'total_vector_count': 10,
 'vector_type': 'dense'}

# Running a query

Next we can run a query
In a more realistic scenario, the ```vector``` values passing into ```query``` would be an embedding vector of something meaningful. but for this simple demo we will use made up values.

```top_k``` specifies the number of results we would like to be returned, the method will return upto ```top_k``` results, but may be less sometime if there are less mathces in the DB

In [ ]:
query_embedding = [2.0, 2.0, 2.0]

index.query(vector=query_embedding, top_k = 5, include_values = True)





{'matches': [{'id': 'id-4',
              'score': 0.993730247,
              'values': [0.300670862, 0.237244204, 0.307836741]},
             {'id': 'id-6',
              'score': 0.976593912,
              'values': [0.200524926, 0.155633822, 0.115717337]},
             {'id': 'id-2',
              'score': 0.936762214,
              'values': [0.434983641, 0.364147723, 0.823387921]},
             {'id': 'id-3',
              'score': 0.910450518,
              'values': [0.769627, 0.21730949, 0.825789]},
             {'id': 'id-9',
              'score': 0.885155618,
              'values': [0.090756692, 0.492109954, 0.475611478]}],
 'namespace': '',
 'usage': {'read_units': 6}}

In [ ]:
# Deleting the index

pc.delete_index(name = index_name)